In [1]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
BUCKET = "gs://retailsmart-lake-retailsmart-bd"
spark.conf.set("spark.sql.catalog.retailsmart", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.retailsmart.type", "hadoop")
spark.conf.set("spark.sql.catalog.retailsmart.warehouse", f"{BUCKET}/lakehouse")
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.sql("CREATE NAMESPACE IF NOT EXISTS retailsmart.smart")

col = F.col
U, S = "retailsmart.universal", "retailsmart.smart"

def guardar(df, nombre):
    t = f"{S}.gold_comercial_{nombre}_v1"
    df.withColumn("_cargado_en", F.current_timestamp()).writeTo(t).using("iceberg").createOrReplace()
    print(nombre, spark.table(t).count(), "filas")

In [2]:
f = spark.sql("SELECT explode(sequence(to_date('2009-01-01'), to_date('2027-12-31'), interval 1 day)) AS fecha")
dim_fecha = (f.withColumn("fecha_sk", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha")).withColumn("trimestre", F.quarter("fecha"))
    .withColumn("mes", F.month("fecha")).withColumn("dia", F.dayofmonth("fecha"))
    .withColumn("dia_semana", F.dayofweek("fecha"))
    .withColumn("es_fin_de_semana", F.dayofweek("fecha").isin(1, 7)))
guardar(dim_fecha, "dim_fecha")

canales = [(1, "thelook", "web"), (2, "olist", "marketplace"), (3, "uci", "tienda_mayorista")]
guardar(spark.createDataFrame(canales, ["canal_sk", "fuente", "canal"]), "dim_canal")

dim_fecha 6939 filas


dim_canal 3 filas


In [10]:
def unidad_olist(cat):
    c = F.lower(cat)
    return (F.when(c == "pet_shop", "Mascotas")
     .when(c == "sin_categoria", "Sin categoría")
     .when(c.rlike("cozinha"), "Hogar y decoración")
     .when(c.rlike("appliance|electronic|computer|audio|telephony|tablet|console|cine_photo|signaling|air_cond|pc_gamer"), "Electrónica y electrodomésticos")
     .when(c.rlike("fashi|underwear|luggage|watches"), "Moda y accesorios")
     .when(c.rlike("health|beauty|perfumery|diaper|baby"), "Belleza, salud y bebés")
     .when(c.rlike("food|drink"), "Alimentos y bebidas")
     .when(c.rlike("construction|costruction|tools|auto|industry|agro|security"), "Construcción, auto e industria")
     .when(c.rlike("sport|leisure|toy|music|cds|dvds|book|art|party|christmas|cool_stuff|flowers|stationery"), "Ocio, deportes y cultura")
     .when(c.rlike("furniture|bed_bath|housewares|home|decor|kitchen|la_cuisine|garden"), "Hogar y decoración")
     .otherwise("Otros"))

olist = (spark.table(f"{U}.silver_logistica_productos_v1").alias("p")
    .join(spark.table(f"{U}.silver_logistica_categorias_v1").alias("c"), "product_category_name", "left")
    .select(F.lit("olist").alias("fuente"), col("product_id").alias("id_origen"),
            F.lit(None).cast("string").alias("nombre"),
            F.coalesce(col("product_category_name_english"), col("product_category_name"),
                       F.lit("sin_categoria")).alias("categoria_origen"),
            F.lit(None).cast("string").alias("marca")))
olist = olist.withColumn("unidad_negocio", unidad_olist(col("categoria_origen")))

thelook = (spark.table(f"{U}.silver_ecommerce_productos_v1")
    .select(F.lit("thelook").alias("fuente"), col("id").cast("string").alias("id_origen"),
            col("name").alias("nombre"), col("category").alias("categoria_origen"),
            col("brand").alias("marca"), F.lit("Moda y accesorios").alias("unidad_negocio")))

uci = (spark.table(f"{U}.silver_tienda_ventas_v1").groupBy("codigo_producto")
    .agg(F.first("descripcion", ignorenulls=True).alias("nombre"))
    .select(F.lit("uci").alias("fuente"), col("codigo_producto").alias("id_origen"), col("nombre"),
            F.lit("sin_categoria").alias("categoria_origen"), F.lit(None).cast("string").alias("marca"),
            F.lit("Hogar y decoración").alias("unidad_negocio")))

dim_producto = (olist.unionByName(thelook).unionByName(uci)
    .withColumn("producto_sk", F.xxhash64(col("fuente"), col("id_origen"))))
guardar(dim_producto, "dim_producto")
dim_producto.groupBy("fuente", "unidad_negocio").count().orderBy("fuente", F.desc("count")).show(40, False)

dim_producto 66009 filas


+-------+-------------------------------+-----+
|fuente |unidad_negocio                 |count|
+-------+-------------------------------+-----+
|olist  |Hogar y decoración             |8771 |
|olist  |Ocio, deportes y cultura       |6830 |
|olist  |Electrónica y electrodomésticos|4790 |
|olist  |Belleza, salud y bebés         |4243 |
|olist  |Construcción, auto e industria |3718 |
|olist  |Moda y accesorios              |2899 |
|olist  |Mascotas                       |719  |
|olist  |Sin categoría                  |610  |
|olist  |Alimentos y bebidas            |267  |
|olist  |Otros                          |104  |
|thelook|Moda y accesorios              |29120|
|uci    |Hogar y decoración             |3938 |
+-------+-------------------------------+-----+



In [4]:
olist_c = (spark.table(f"{U}.silver_logistica_clientes_v1").groupBy("customer_unique_id")
    .agg(F.first("customer_state").alias("estado_origen"), F.first("customer_city").alias("ciudad"))
    .select(F.lit("olist").alias("fuente"), col("customer_unique_id").alias("id_origen"),
            F.lit("Brasil").alias("pais_origen"), "estado_origen", "ciudad"))

thelook_c = (spark.table(f"{U}.silver_ecommerce_usuarios_v1")
    .select(F.lit("thelook").alias("fuente"), col("id").cast("string").alias("id_origen"),
            col("country").alias("pais_origen"), col("state").alias("estado_origen"), col("city").alias("ciudad"),
            col("age").alias("edad"), col("gender").alias("genero"),
            F.sha2(col("email"), 256).alias("email_hash"), col("created_at").alias("fecha_alta")))

uci_c = (spark.table(f"{U}.silver_tienda_ventas_v1").filter(col("cliente_id").isNotNull())
    .groupBy("cliente_id").agg(F.first("pais").alias("pais_origen"))
    .select(F.lit("uci").alias("fuente"), col("cliente_id").alias("id_origen"), "pais_origen"))

dim_cliente = (olist_c.unionByName(thelook_c, allowMissingColumns=True)
    .unionByName(uci_c, allowMissingColumns=True)
    .withColumn("cliente_sk", F.xxhash64(col("fuente"), col("id_origen"))))
guardar(dim_cliente, "dim_cliente")

dim_cliente 200467 filas


In [5]:
print("== países thelook")
spark.table(f"{U}.silver_ecommerce_usuarios_v1").groupBy("country").count().orderBy(F.desc("count")).show(30, False)
print("== países UCI")
spark.table(f"{U}.silver_tienda_ventas_v1").groupBy("pais").count().orderBy(F.desc("count")).show(60, False)
print("== estados thelook en Brasil")
spark.table(f"{U}.silver_ecommerce_usuarios_v1").filter(col("country").isin("Brasil", "Brazil")).groupBy("state").count().orderBy(F.desc("count")).show(40, False)
print("== estados Olist")
spark.table(f"{U}.silver_logistica_clientes_v1").groupBy("customer_state").count().orderBy(F.desc("count")).show(30, False)
print("== categorías thelook")
spark.table(f"{U}.silver_ecommerce_productos_v1").groupBy("category").count().orderBy(F.desc("count")).show(40, False)

== países thelook


+--------------+-----+
|country       |count|
+--------------+-----+
|China         |33904|
|United States |22494|
|Brasil        |14562|
|South Korea   |5380 |
|France        |4760 |
|United Kingdom|4643 |
|Germany       |4169 |
|Spain         |3940 |
|Japan         |2450 |
|Australia     |2193 |
|Belgium       |1221 |
|Poland        |259  |
|Colombia      |21   |
|España        |3    |
|Austria       |1    |
+--------------+-----+

== países UCI


+--------------------+------+
|pais                |count |
+--------------------+------+
|United Kingdom      |487806|
|Germany             |9478  |
|France              |8540  |
|EIRE                |8180  |
|Spain               |2527  |
|Netherlands         |2367  |
|Belgium             |2069  |
|Switzerland         |1993  |
|Portugal            |1510  |
|Australia           |1255  |
|Norway              |1085  |
|Italy               |803   |
|Channel Islands     |757   |
|Finland             |695   |
|Cyprus              |611   |
|Sweden              |461   |
|Unspecified         |442   |
|Austria             |401   |
|Denmark             |389   |
|Japan               |358   |
|Poland              |341   |
|Israel              |294   |
|USA                 |291   |
|Hong Kong           |284   |
|Singapore           |229   |
|Iceland             |182   |
|Canada              |151   |
|Greece              |146   |
|Malta               |127   |
|United Arab Emirates|68    |
|European 

+-------------------+-----+
|state              |count|
+-------------------+-----+
|São Paulo          |2196 |
|Minas Gerais       |1532 |
|Bahia              |1382 |
|Pará               |1273 |
|Maranhão           |1011 |
|Ceará              |969  |
|Rio de Janeiro     |881  |
|Pernambuco         |680  |
|Rio Grande do Sul  |650  |
|Paraná             |562  |
|Amazonas           |562  |
|Distrito Federal   |437  |
|Santa Catarina     |348  |
|Goiás              |309  |
|Mato Grosso        |293  |
|Rio Grande do Norte|227  |
|Alagoas            |192  |
|Sergipe            |172  |
|Mato Grosso do Sul |158  |
|Espírito Santo     |149  |
|Piauí              |138  |
|Rondônia           |120  |
|Amapá              |108  |
|Paraíba            |93   |
|Acre               |55   |
|Tocantins          |47   |
|Roraima            |18   |
+-------------------+-----+

== estados Olist


+--------------+-----+
|customer_state|count|
+--------------+-----+
|SP            |41746|
|RJ            |12852|
|MG            |11635|
|RS            |5466 |
|PR            |5045 |
|SC            |3637 |
|BA            |3380 |
|DF            |2140 |
|ES            |2033 |
|GO            |2020 |
|PE            |1652 |
|CE            |1336 |
|PA            |975  |
|MT            |907  |
|MA            |747  |
|MS            |715  |
|PB            |536  |
|PI            |495  |
|RN            |485  |
|AL            |413  |
|SE            |350  |
|TO            |280  |
|RO            |253  |
|AM            |148  |
|AC            |81   |
|AP            |68   |
|RR            |46   |
+--------------+-----+

== categorías thelook


+-----------------------------+-----+
|category                     |count|
+-----------------------------+-----+
|Intimates                    |2363 |
|Jeans                        |1999 |
|Tops & Tees                  |1868 |
|Fashion Hoodies & Sweatshirts|1866 |
|Swim                         |1798 |
|Sleep & Lounge               |1771 |
|Shorts                       |1765 |
|Sweaters                     |1737 |
|Accessories                  |1559 |
|Active                       |1432 |
|Outerwear & Coats            |1420 |
|Underwear                    |1088 |
|Pants                        |1041 |
|Dresses                      |955  |
|Socks                        |905  |
|Maternity                    |898  |
|Plus                         |758  |
|Suits & Sport Coats          |739  |
|Socks & Hosiery              |666  |
|Pants & Capris               |613  |
|Leggings                     |564  |
|Blazers & Jackets            |561  |
|Skirts                       |367  |
|Suits      

In [6]:
from itertools import chain
UF_NOMBRE = {"AC":"Acre","AL":"Alagoas","AP":"Amapá","AM":"Amazonas","BA":"Bahia","CE":"Ceará",
 "DF":"Distrito Federal","ES":"Espírito Santo","GO":"Goiás","MA":"Maranhão","MT":"Mato Grosso",
 "MS":"Mato Grosso do Sul","MG":"Minas Gerais","PA":"Pará","PB":"Paraíba","PR":"Paraná",
 "PE":"Pernambuco","PI":"Piauí","RJ":"Rio de Janeiro","RN":"Rio Grande do Norte",
 "RS":"Rio Grande do Sul","RO":"Rondônia","RR":"Roraima","SC":"Santa Catarina","SP":"São Paulo",
 "SE":"Sergipe","TO":"Tocantins"}
NOMBRE_UF = {v: k for k, v in UF_NOMBRE.items()}
PAIS_ALIAS = {"Brasil":"Brazil","España":"Spain","USA":"United States","EIRE":"Ireland","RSA":"South Africa"}
mapa_alias = F.create_map(*[F.lit(x) for x in chain(*PAIS_ALIAS.items())])
mapa_uf = F.create_map(*[F.lit(x) for x in chain(*NOMBRE_UF.items())])

NORTE, NORDESTE = ["AC","AP","AM","PA","RO","RR","TO"], ["AL","BA","CE","MA","PB","PE","PI","RN","SE"]
CENTRO, SUDESTE, SUR = ["DF","GO","MT","MS"], ["ES","MG","RJ","SP"], ["PR","RS","SC"]
NORTEAMERICA = ["United States","Canada"]
EUROPA = ["United Kingdom","Germany","France","Ireland","Spain","Netherlands","Belgium","Switzerland",
 "Portugal","Norway","Italy","Channel Islands","Finland","Cyprus","Sweden","Austria","Denmark","Poland",
 "Iceland","Greece","Malta","Lithuania","Czech Republic","European Community"]

def geo(pais, estado):
    p = F.coalesce(mapa_alias[F.trim(pais)], F.trim(pais))
    uf = F.when(F.length(F.trim(estado)) == 2, F.upper(F.trim(estado))).otherwise(mapa_uf[F.trim(estado)])
    e = F.when(p == "Brazil", F.coalesce(uf, F.lit("ND"))).otherwise(F.lit("N/A"))
    return p, e

def region(p, e):
    br = p == "Brazil"
    return (F.when(br & e.isin(*NORTE), "Brasil Norte").when(br & e.isin(*NORDESTE), "Brasil Nordeste")
        .when(br & e.isin(*CENTRO), "Brasil Centro-Oeste").when(br & e.isin(*SUDESTE), "Brasil Sudeste")
        .when(br & e.isin(*SUR), "Brasil Sur").when(br | (p == "Unspecified"), "Sin asignar")
        .when(p.isin(*NORTEAMERICA), "Norteamérica").when(p.isin(*EUROPA), "Europa")
        .otherwise("Asia-Pacífico y resto"))

raw_geo = (spark.table(f"{U}.silver_logistica_clientes_v1").select(F.lit("Brazil").alias("pais_raw"), col("customer_state").alias("estado_raw"))
    .unionByName(spark.table(f"{U}.silver_ecommerce_usuarios_v1").select(col("country").alias("pais_raw"), col("state").alias("estado_raw")))
    .unionByName(spark.table(f"{U}.silver_tienda_ventas_v1").select(col("pais").alias("pais_raw"), F.lit(None).cast("string").alias("estado_raw")))
    .distinct())
p, e = geo(col("pais_raw"), col("estado_raw"))
dim_geo = (raw_geo.select(p.alias("pais"), e.alias("estado")).distinct()
    .withColumn("region", region(col("pais"), col("estado")))
    .withColumn("geo_sk", F.xxhash64(col("pais"), col("estado"))))
guardar(dim_geo, "dim_geo")
dim_geo.groupBy("region").count().orderBy("region").show(truncate=False)

dim_geo 68 filas


+---------------------+-----+
|region               |count|
+---------------------+-----+
|Asia-Pacífico y resto|13   |
|Brasil Centro-Oeste  |4    |
|Brasil Nordeste      |9    |
|Brasil Norte         |7    |
|Brasil Sudeste       |4    |
|Brasil Sur           |3    |
|Europa               |24   |
|Norteamérica         |2    |
|Sin asignar          |2    |
+---------------------+-----+



In [7]:
assert "fechas_inconsistentes" in spark.table(f"{U}.silver_ecommerce_items_pedido_v1").columns, \
    "Ejecuta primero la reescritura de items_pedido en el notebook 04"

cl = lambda fuente, x: F.xxhash64(F.lit(fuente), x.cast("string"))
fsk = lambda d: F.date_format(d, "yyyyMMdd").cast("int")

# thelook: cada fila de ítem es una unidad vendida
oi = spark.table(f"{U}.silver_ecommerce_items_pedido_v1").select(
    col("id").cast("string").alias("id_linea"), "order_id", "product_id",
    col("status").alias("estado"), "sale_price", "fechas_inconsistentes")
o = spark.table(f"{U}.silver_ecommerce_pedidos_v1").select("order_id", "user_id", col("created_at").alias("fecha"))
u = spark.table(f"{U}.silver_ecommerce_usuarios_v1").select(
    col("id").alias("user_id"), col("country").alias("pais_raw"), col("state").alias("estado_raw"))
p, e = geo(col("pais_raw"), col("estado_raw"))
f_thelook = (oi.join(o, "order_id").join(u, "user_id").select(
    F.lit("thelook").alias("fuente"), col("order_id").cast("string").alias("id_pedido"), "id_linea",
    fsk(col("fecha")).alias("fecha_sk"), cl("thelook", col("product_id")).alias("producto_sk"),
    cl("thelook", col("user_id")).alias("cliente_sk"), F.xxhash64(p, e).alias("geo_sk"),
    F.lit(1).alias("cantidad"), col("sale_price").alias("monto"),
    F.lit(None).cast("double").alias("costo_envio"), F.lit("USD").alias("moneda"),
    "estado", "fechas_inconsistentes"))

# Olist: cliente por persona (customer_unique_id), ubicación por estado
it = spark.table(f"{U}.silver_logistica_items_pedido_v1").select(
    "order_id", col("order_item_id").cast("string").alias("id_linea"), "product_id",
    col("price").alias("monto"), col("freight_value").alias("costo_envio"))
od = spark.table(f"{U}.silver_logistica_pedidos_v1").select(
    "order_id", "customer_id", col("order_status").alias("estado"), col("order_purchase_timestamp").alias("fecha"))
cu = spark.table(f"{U}.silver_logistica_clientes_v1").select(
    "customer_id", col("customer_unique_id").alias("uid"), col("customer_state").alias("estado_raw"))
p, e = geo(F.lit("Brazil"), col("estado_raw"))
f_olist = (it.join(od, "order_id").join(cu, "customer_id").select(
    F.lit("olist").alias("fuente"), col("order_id").alias("id_pedido"), "id_linea",
    fsk(col("fecha")).alias("fecha_sk"), cl("olist", col("product_id")).alias("producto_sk"),
    cl("olist", col("uid")).alias("cliente_sk"), F.xxhash64(p, e).alias("geo_sk"),
    F.lit(1).alias("cantidad"), "monto", "costo_envio", F.lit("BRL").alias("moneda"),
    "estado", F.lit(False).alias("fechas_inconsistentes")))

# UCI: el monto de las cancelaciones es negativo (cantidad negativa por precio)
v = spark.table(f"{U}.silver_tienda_ventas_v1")
p, e = geo(col("pais"), F.lit(None).cast("string"))
f_uci = v.select(
    F.lit("uci").alias("fuente"), col("factura").alias("id_pedido"),
    F.xxhash64("factura", "codigo_producto", "fecha_factura", "cantidad", "precio_unitario", "cliente_id").cast("string").alias("id_linea"),
    fsk(col("fecha_factura")).alias("fecha_sk"), cl("uci", col("codigo_producto")).alias("producto_sk"),
    F.when(col("cliente_id").isNotNull(), cl("uci", col("cliente_id"))).alias("cliente_sk"),
    F.xxhash64(p, e).alias("geo_sk"), col("cantidad"), (col("cantidad") * col("precio_unitario")).alias("monto"),
    F.lit(None).cast("double").alias("costo_envio"), F.lit("GBP").alias("moneda"),
    F.when(col("es_cancelacion"), "cancelado").otherwise("completado").alias("estado"),
    F.lit(False).alias("fechas_inconsistentes"))

fact = (f_thelook.unionByName(f_olist).unionByName(f_uci)
    .withColumn("canal_sk", F.when(col("fuente") == "thelook", 1).when(col("fuente") == "olist", 2).otherwise(3)))
guardar(fact, "fact_ventas")

fact_ventas 828423 filas


In [8]:
ft = spark.table(f"{S}.gold_comercial_fact_ventas_v1")
ft.groupBy("fuente", "moneda").agg(F.count("*").alias("filas"), F.round(F.sum("monto"), 2).alias("monto_total")).show()
for dim, k in [("dim_producto", "producto_sk"), ("dim_geo", "geo_sk"), ("dim_fecha", "fecha_sk"), ("dim_cliente", "cliente_sk")]:
    d = spark.table(f"{S}.gold_comercial_{dim}_v1").select(k)
    print(dim, "filas huérfanas:", ft.filter(col(k).isNotNull()).join(d, k, "left_anti").count())

+-------+------+------+-------------+
| fuente|moneda| filas|  monto_total|
+-------+------+------+-------------+
|  olist|   BRL|112650| 1.35916437E7|
|thelook|   USD|181644|1.084360601E7|
|    uci|   GBP|534129|   9748131.07|
+-------+------+------+-------------+



dim_producto filas huérfanas: 0


dim_geo filas huérfanas: 0


dim_fecha filas huérfanas: 0


dim_cliente filas huérfanas: 0


In [9]:
spark.table(f"{S}.gold_comercial_dim_producto_v1").filter((col("fuente") == "olist") & (col("unidad_negocio") == "Otros")).groupBy("categoria_origen").count().orderBy(F.desc("count")).show(30, False)

+---------------------------------------------+-----+
|categoria_origen                             |count|
+---------------------------------------------+-----+
|pet_shop                                     |719  |
|sin_categoria                                |610  |
|market_place                                 |104  |
|portateis_cozinha_e_preparadores_de_alimentos|10   |
+---------------------------------------------+-----+



In [11]:
spark.sql(f"""
SELECT g.region, p.unidad_negocio, f.moneda, COUNT(*) AS lineas, ROUND(SUM(f.monto), 0) AS monto
FROM {S}.gold_comercial_fact_ventas_v1 f
JOIN {S}.gold_comercial_dim_geo_v1 g ON f.geo_sk = g.geo_sk
JOIN {S}.gold_comercial_dim_producto_v1 p ON f.producto_sk = p.producto_sk
GROUP BY 1, 2, 3 ORDER BY lineas DESC LIMIT 15""").show(truncate=False)

+---------------------+-------------------------------+------+------+---------+
|region               |unidad_negocio                 |moneda|lineas|monto    |
+---------------------+-------------------------------+------+------+---------+
|Europa               |Hogar y decoración             |GBP   |530594|9532290.0|
|Asia-Pacífico y resto|Moda y accesorios              |USD   |79517 |4743865.0|
|Norteamérica         |Moda y accesorios              |USD   |41124 |2452488.0|
|Europa               |Moda y accesorios              |USD   |34740 |2063304.0|
|Brasil Sudeste       |Hogar y decoración             |BRL   |21495 |2039864.0|
|Brasil Sudeste       |Ocio, deportes y cultura       |BRL   |14290 |1714357.0|
|Brasil Sudeste       |Electrónica y electrodomésticos|BRL   |12829 |1516516.0|
|Brasil Sudeste       |Belleza, salud y bebés         |BRL   |11026 |1318197.0|
|Brasil Nordeste      |Moda y accesorios              |USD   |8844  |526423.0 |
|Brasil Sudeste       |Moda y accesorios

In [12]:
TABLAS = ["dim_fecha", "dim_canal", "dim_producto", "dim_cliente", "dim_geo", "fact_ventas"]
for t in TABLAS:
    (spark.table(f"{S}.gold_comercial_{t}_v1").write.format("bigquery")
        .option("table", f"retailsmart-bd.retailsmart_smart.{t}")
        .option("temporaryGcsBucket", "retailsmart-lake-retailsmart-bd")
        .mode("overwrite").save())
    print(t, "publicada")

dim_fecha publicada


dim_canal publicada


dim_producto publicada


dim_cliente publicada


dim_geo publicada


fact_ventas publicada
